# LAB | API Calling to ChatGPT
## Name: Martha Olukayode
## Project Description: 

You've been tasked with building an automation system that:
- Takes a product image and basic metadata (name, price, category)
- Sends it to ChatGPT with a prompt
- Receives a professionally written product listing
- Formats it for use in your e-commerce platform

### Setup

In [19]:
!pip install openai pydantic python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
# **************************************
# Step 1: Setting Up OpenAI API Access
# **************************************

import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

# Load OPENAI_API_KEY from the .env file next to this notebook
load_dotenv()

# Initialize client (reads OPENAI_API_KEY from the environment)
api_key = os.getenv("OPENAI_API_KEY")

# Check whether Python found the key
if api_key:
    print("✓ API key loaded successfully!")
else:
    print("✗ API key was not found.")

client = OpenAI(api_key=api_key)

print("✓ OpenAI client initialized successfully!")


✓ API key loaded successfully!
✓ OpenAI client initialized successfully!


In [81]:
# **************************************
# Step 2: Preparing the Dataset
# **************************************
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path

# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    # Read the CSV file
    products_df = pd.read_csv("data/styles.csv")
    print(f"✓ Loaded {len(products_df)} products")

    print("\n✓ Dataset prepared!")
    print(f"  Total products: {len(products_df)}")



# Create list for products
products = []

# Process each row in the DataFrame
for index, pro in products_df.iterrows():
    product = {}

    product["id"] = pro["id"]
    product["name"] = pro["productDisplayName"]
# styles.csv may not contain a price column
    if "price" in products_df.columns:
        product["price"] = pro["price"]
    else:
        product["price"] = 0.00

    product["category"] = pro["subCategory"]

    # Fashion dataset images normally use the product ID as filename
    product["image_path"] = f"product_images/{pro['id']}.jpg"

    products.append(product)
        



Loading product dataset...
✓ Loaded 100 products
⚠ Could not load HuggingFace dataset: To support decoding images, please install 'Pillow'.
Using local images instead...
✓ Loaded 20 products

✓ Dataset prepared!
  Total products: 20


In [22]:
# Sample product information
product = {
    "id": 1,
    "name": "Wireless Headphones",
    "price": 79.99,
    "category": "Electronics",
    "image_path": "product_images/product1.jpg"
}

print("\nProduct Information:")
print(product)


Product Information:
{'id': 1, 'name': 'Wireless Headphones', 'price': 79.99, 'category': 'Electronics', 'image_path': 'product_images/product1.jpg'}


In [23]:
# **************************************
# Step 3: Encoding Images for API
# **************************************
import base64

def encode_image_to_base64(image_path):
    """Convert an image file into a Base64 string."""

    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode('utf-8')

    return encoded_image

# Encode the product image
encoded_image = encode_image_to_base64(product["image_path"])

print("\nImage encoded successfully!")
print("\nProduct Information with Encoded Image:")
print("Encoded image length:", len(encoded_image))
print("First 40 characters:", encoded_image[:40] + "...")


Image encoded successfully!

Product Information with Encoded Image:
Encoded image length: 22236
First 40 characters: /9j/4AAQSkZJRgABAQEASABIAAD/7RV8UGhvdG9z...


In [24]:
# **************************************
# 4: Creating the Product Listing Prompt
# **************************************
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
product_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(product_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [32]:
# **************************************
# Step 5: Calling the ChatGPT API with Vision
# **************************************
import os
import base64
import json
from dotenv import load_dotenv
from openai import OpenAI
# Send the product image and prompt to OpenAI
try:
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": product_prompt
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{encoded_image}"
                    }
                ]
            }
        ]
    )

    print("\n=== AI GENERATED PRODUCT LISTING ===")
    print(response.output_text)

    # Get the AI response
    response_text = response.output_text.strip()

    # Remove Markdown code blocks if the AI added them
    if response_text.startswith("```json"):
        response_text = response_text[7:]

    if response_text.startswith("```"):
        response_text = response_text[3:]

    if response_text.endswith("```"):
        response_text = response_text[:-3]

    response_text = response_text.strip()

    # Convert JSON text into a Python dictionary
    product_listing = json.loads(response_text)

    print("\n✓ Response converted to Python dictionary!")

    # Save the result to a JSON file
    with open("product_listings.json", "w", encoding="utf-8") as file:
        json.dump(product_listing, file, indent=4, ensure_ascii=False)

    print("✓ Product listing saved to product_listings.json")   

except Exception as e:
    print("API request failed:", e)


=== AI GENERATED PRODUCT LISTING ===
```json
{
    "title": "Premium Wireless Bluetooth Headphones - 30H Battery Life",
    "description": "Experience unparalleled sound quality with our Premium Wireless Bluetooth Headphones. Engineered for those who appreciate clarity, these headphones feature advanced noise-cancelling technology, ensuring you stay immersed in your music without distractions. With an astonishing 30-hour battery life, you can enjoy uninterrupted listening all day long, whether you're commuting, working out, or simply relaxing at home. Lightweight and designed for comfort, these headphones sport an adjustable headband and soft ear cushions for an ideal fit during extended use. Plus, with easy Bluetooth connectivity, you can pair them effortlessly with your smartphone or other devices. Elevate your audio experience today!",
    "features": [
        "Advanced Noise Cancelling Technology",
        "30-Hour Long Battery Life",
        "Lightweight and Comfortable Design",

In [36]:
def generate_product_listing(product):
    """Generate an AI product listing for one product."""

    try:
        print(f"\nProcessing: {product['name']}...")

        # 1. Encode the product image
        encoded_image = encode_image_to_base64(
            product["image_path"]
        )

        # 2. Create the prompt
        product_prompt = create_product_listing_prompt(
            product["name"],
            product["price"],
            product["category"]
        )

        # 3. Send prompt and image to OpenAI
        response = client.responses.create(
            model="gpt-4o-mini",
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": product_prompt
                        },
                        {
                            "type": "input_image",
                            "image_url": (
                                f"data:image/jpeg;base64,"
                                f"{encoded_image}"
                            )
                        }
                    ]
                }
            ]
        )

        # 4. Get the response text
        response_text = response.output_text.strip()

        # 5. Remove Markdown formatting if present
        if response_text.startswith("```json"):
            response_text = response_text[7:]

        if response_text.startswith("```"):
            response_text = response_text[3:]

        if response_text.endswith("```"):
            response_text = response_text[:-3]

        response_text = response_text.strip()

        # 6. Convert JSON text into a Python dictionary
        listing = json.loads(response_text)

        print(f"✓ Listing generated for {product['name']}")

        return listing

    except Exception as e:
        print(f"✗ Error processing {product['name']}: {e}")
        return None

In [ ]:
# **************************************
# Step 6: Processing Multiple Products
# **************************************


In [50]:
# Store all successful product listings
all_listings = []

# Process each product
for product in products:

    listing = generate_product_listing(product)

    if listing is not None:

        # Add original product information
        listing["product_id"] = product["id"]
        listing["original_name"] = product["name"]
        listing["price"] = product["price"]
        listing["category"] = product["category"]

        # Add the listing to our results
        all_listings.append(listing)


# Save all listings to one JSON file
with open("product_listings.json", "w", encoding="utf-8") as file:

    json.dump(
        all_listings,
        file,
        indent=4,
        ensure_ascii=False
    )


print("\n=================================")
print("PROCESSING COMPLETE")
print("=================================")
print(f"Successful listings: {len(all_listings)}")
print("Results saved to product_listings.json")


Processing: Wireless Headphones...
✓ Listing generated for Wireless Headphones

Processing: Running Shoes...
✓ Listing generated for Running Shoes

Processing: Travel Backpack...
✓ Listing generated for Travel Backpack

Processing: Test Product...
✗ Error processing Test Product: [Errno 2] No such file or directory: 'product_images/image_does_not_exist.jpg'

PROCESSING COMPLETE
Successful listings: 3
Results saved to product_listings.json
